# Module 1: Deep Agents

> Part of the **Modular Workshops** series. Standalone, ~45 min.

Deep Agents = `create_agent()` + a pre-built middleware stack (filesystem, planning, subagents, context management). We'll build up from a bare agent to a fully-featured **in-store shopping assistant** — the kind of helper a shopper talks to while walking the aisles — exploring:

- The harness and built-in tools
- Custom tools (a store aisle + stock directory) alongside native web search
- Subagents and context isolation
- Backends and persistent memory
- Middleware (compliance, audit)
- Human-in-the-loop on tool calls
- AGENTS.md and Skills


## Setup

In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.models import model

from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
from langsmith import uuid7
from IPython.display import Image, display

# Start each run with a clean long-term memory store (section 1.4 recreates it).
for f in Path().glob("deep_agents_memory.db*"):
    f.unlink()

print("Ready")

---
# Part 1: Deep Agents

Deep Agents = `create_agent()` + a pre-built middleware stack (filesystem, planning, subagents, context management).

We'll build up from a bare agent to a fully-featured **in-store shopping assistant** that helps a shopper find items, check stock, and work through their list aisle by aisle.

## 1.1 Your First Deep Agent

`create_deep_agent()` gives you a filesystem, a todo list, and context management out of the box — no tools required.

<img src="../images/deepAgentsDiag.png" style="width: auto; max-height: 420px; border-radius: 8px;">

### What you get for free:

- **Filesystem Tools** — `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`
- **Planning Tool** — `write_todos` for task tracking
- **Subagent Delegation** — `task()` tool for isolated work
- **Large Tool Result Eviction** — Automatically offloads tool results >20k tokens to the filesystem
- **Conversation Summarization** — Compresses history when approaching ~85% context capacity
- **Dangling Tool Call Patching** — Fixes message history consistency automatically


In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model,
    system_prompt="You are a helpful assistant.",
    checkpointer=MemorySaver(),
)
agent

In [ ]:
# The agent can already write and read files — these are built-in tools
config = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "I just walked into the store to shop for taco night. Write my shopping list to /store_list.txt, then read it back to me."}]
}, config=config)

for m in result["messages"]:
    m.model_copy(update={"content": m.text}).pretty_print()

In [ ]:
# Helper: print the virtual filesystem from a deep agent result.
def print_files(result, header="VIRTUAL FILESYSTEM (in-memory, not on disk!)"):
    files = result.get("files") or {}
    if not files:
        print("(no files in state)")
        return
    print("=" * 50)
    print(header)
    print("=" * 50)
    for path, file_data in files.items():
        print(f"\n  Path: {path!r}")
        print("  " + "-" * 38)
        content = file_data
        if isinstance(file_data, dict) and "content" in file_data:
            content = file_data["content"]
        if isinstance(content, list):
            content = "\n".join(content)
        for line in str(content).split("\n"):
            print(f"  | {line}")

print_files(result)


### Filesystem persistence within a thread

By default, `create_deep_agent()` uses **StateBackend** — files are stored in agent state and persist within a thread (via the checkpointer), but disappear when you start a new thread.

| Backend | Storage | Persistence | Use Case |
|---------|---------|-------------|----------|
| **StateBackend** | In-memory (agent state) | Single thread | Scratch pads, intermediate results |
| **FilesystemBackend** | Local disk | Permanent | Direct file access (use with caution) |
| **StoreBackend** | LangGraph Store | Cross-thread | Long-term memories |
| **CompositeBackend** | Routes to others | Mixed | Selective persistence |

In [ ]:
# Same thread — the file persists via the checkpointer
result = agent.invoke({
    "messages": [{"role": "user", "content": "Read the file /store_list.txt"}]
}, config=config)

print("Same thread:\n\n", result["messages"][-1].text)

In [ ]:
# New thread — StateBackend is ephemeral, so the file is gone
new_config = {"configurable": {"thread_id": str(uuid7())}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "List all files with ls /"}]
}, config=new_config)

print("New thread:", result["messages"][-1].text)

### Key Takeaway
- `create_deep_agent()` gives you filesystem + planning capabilities for free
- Files are stored in agent state (virtual, not on disk)
- `StateBackend` (default) persists within a thread but is ephemeral across threads
- We'll see how to make files persist across threads with `CompositeBackend` + `StoreBackend` in section 1.4

## 1.2 Custom Tools

Add your own tools alongside the built-in ones. An in-store assistant needs two kinds of lookups:

- **Store-specific data** — which aisle an item is in and whether it's in stock. This lives in the store's own systems, so we expose it as a custom `@tool` (`store_directory`) backed by a small mock catalog.
- **Open-ended data** — recipes, general product info, substitution ideas. For this we add the **model provider's native web search** (`{'type': 'web_search'}`), which the model runs server-side and returns with citations.

The agent picks the right tool for each question — aisle/stock from the store directory, everything else from web search.

In [ ]:
# A store-specific tool: aisle location + live stock for this store.
# Backed by a small mock catalog — in production this would hit the store's
# inventory / planogram API.
STORE_CATALOG = {
    "milk": {"aisle": "D2 (Dairy)", "in_stock": True, "price": 3.49},
    "eggs": {"aisle": "D1 (Dairy)", "in_stock": True, "price": 2.99},
    "tortillas": {"aisle": "7 (Bakery/Intl)", "in_stock": True, "price": 2.49},
    "ground beef": {"aisle": "M3 (Meat)", "in_stock": False, "price": 6.99},
    "black beans": {"aisle": "5 (Canned)", "in_stock": True, "price": 1.29},
    "shredded cheese": {"aisle": "D3 (Dairy)", "in_stock": True, "price": 3.99},
    "salsa": {"aisle": "6 (Condiments)", "in_stock": True, "price": 2.79},
    "avocado": {"aisle": "1 (Produce)", "in_stock": True, "price": 1.19},
    "lettuce": {"aisle": "1 (Produce)", "in_stock": True, "price": 1.49},
    "tomatoes": {"aisle": "1 (Produce)", "in_stock": True, "price": 0.99},
}

@tool(parse_docstring=True)
def store_directory(item: str) -> str:
    """Look up an item's aisle, stock status, and price in this store.

    Args:
        item: The grocery item to locate (e.g. "milk", "ground beef").
    """
    entry = STORE_CATALOG.get(item.strip().lower())
    if entry is None:
        return f"'{item}' not found in this store's directory. Try a simpler name or ask an associate."
    stock = "in stock" if entry["in_stock"] else "OUT OF STOCK"
    return f"{item}: aisle {entry['aisle']}, {stock}, ${entry['price']:.2f}"

# The model provider's native web search — a built-in Responses API tool for
# open-ended lookups (recipes, general product info, substitution ideas).
web_search = {"type": "web_search"}

agent = create_deep_agent(
    model=model,
    tools=[store_directory, web_search],
    system_prompt=(
        "You are an in-store shopping assistant. Use store_directory for aisle, "
        "stock, and price questions about this store, and web search for recipes "
        "or general product info. Help the shopper move through the store efficiently."
    ),
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "I'm making tacos. Which aisles do I need for tortillas, ground beef, and salsa - and is everything in stock? Save the aisle-by-aisle list to /store_route.md"}]
}, config=config)

print("Agent reply:", result["messages"][-1].text)

In [ ]:
# Print state of virtual file system
print_files(result)

## 1.3 Subagents: Isolated Delegation

Subagents run in a separate context. The main assistant delegates via `task()` and only sees the final result — keeping the main context clean while a helper does the per-item legwork (aisle, stock, substitution).

<img src="../images/deepAgentSubagents.png" style="width: auto; max-height: 380px; border-radius: 8px;">

In [ ]:
from datetime import datetime

item_finder_subagent = {
    "name": "item-finder-agent",
    "description": "Locate one item in the store: aisle, stock, price, and a substitution if it's out of stock. Give one item at a time.",
    "system_prompt": f"""You are an in-store item finder. Today is {datetime.now().strftime('%Y-%m-%d')}.
Use store_directory to get the aisle, stock status, and price. If the item is out of stock,
use web search to suggest one practical substitution the shopper can grab instead.
Report concisely: aisle, stock, price, and (if needed) a substitution.""",
    "tools": [store_directory, web_search],
}

agent = create_deep_agent(
    model=model,
    system_prompt="""You are an in-store shopping coordinator.
For each item on the shopper's list, delegate to the item-finder-agent using the task() tool.
Then assemble an aisle-ordered route and flag anything out of stock.""",
    subagents=[item_finder_subagent],
    checkpointer=MemorySaver(),
)
agent

In [ ]:
config = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "I'm in the store with this list: ground beef, tortillas, salsa. Find each item's aisle and stock, and tell me what to do about anything that's out."}]
}, config=config)

def truncate(text, limit=1000):
    return text if len(text) <= limit else text[:limit] + "…"

for m in result["messages"]:
    m.model_copy(update={"content": truncate(m.text)}).pretty_print()

## 1.4 Backends & Memory

By default, files live in ephemeral state (`StateBackend`). Use `CompositeBackend` to route paths — e.g. `/memories/` to persistent `StoreBackend` while everything else stays ephemeral.

`StoreBackend` is a database-backed store meant to persist memory across threads. Here we back it with a local SQLite file, but the LangGraph `Store` can be backed by the database of your choice (Postgres, etc.).

`StoreBackend` scopes what it reads and writes by `namespace` — a required argument, and your lever for isolation: per user, per assistant, or shared across everyone as here.


In [ ]:
import sqlite3
from deepagents.backends import StateBackend, StoreBackend, CompositeBackend
from langgraph.store.sqlite import SqliteStore

MEMORY_DB = "deep_agents_memory.db"

def open_memory_store(path=MEMORY_DB):
    """Open (or create) a persistent SQLite-backed long-term memory store."""
    conn = sqlite3.connect(path, check_same_thread=False, isolation_level=None)
    store = SqliteStore(conn)
    store.setup()  # creates tables IF NOT EXISTS -> safe whether or not the DB exists
    return store

store = open_memory_store()

# Pass a CompositeBackend *instance* (not a factory)
backend = CompositeBackend(
    default=StateBackend(),                                  # ephemeral scratch space
    routes={
        "/memories/": StoreBackend(                          # persists across threads (SQLite on disk)
            store=store,
            namespace=lambda rt: ("memories", "shared"),
        ),
    },
)

agent = create_deep_agent(
    model=model,
    tools=[store_directory, web_search],
    system_prompt=(
        "You are an in-store shopping assistant. Save durable shopper details "
        "(home store, dietary needs, favorite brands, loyalty ID) to /memories/ for future trips. "
        "ALWAYS check /memories files before answering so you tailor aisles and picks to this shopper."
    ),
    subagents=[item_finder_subagent],
    backend=backend,
    store=store,
    checkpointer=MemorySaver(),
)

# Thread 1: agent saves a memory
config1 = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "For future trips: my home store is the Riverside location, I'm vegetarian, and I shop for a household of 3. Save this to /memories/shopper.md"}]
}, config=config1)
print("Thread 1:", result["messages"][-1].text)

In [ ]:
# Thread 2: different thread, but /memories/ persists via StoreBackend
config2 = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "Which store am I in and what should you keep in mind while I shop? Check /memories/"}]
}, config=config2)
print("Thread 2:", result["messages"][-1].text)

## 1.5 Middleware: Pluggable Behavior

Middleware hooks into `wrap_model_call` (every LLM call) and `wrap_tool_call` (every tool call). This lets you inject rules, audit, or intercept without changing agent code.

<img src="../images/deepAgentMiddleware.png" style="width: auto; max-height: 380px; border-radius: 8px;">

### Built-in context management

Three strategies the deep-agent middleware uses to keep within the model's context window:

<img src="../images/Offloading Inputs LangChain.png" style="width: auto; max-height: 240px; border-radius: 8px;">

**Offload Large Inputs** — file write/edit tool calls leave the full content in conversation history. At ~85% context capacity, deep agents truncate older tool calls and replace them with a file-pointer reference.

<img src="../images/Offloading Results LangChain.png" style="width: auto; max-height: 240px; border-radius: 8px;">

**Offload Large Results** — tool results over ~20k tokens are written to the backend and swapped with a path + 10-line preview. The agent can re-read or grep the full content as needed.

<img src="../images/LangChain Summarization.png" style="width: auto; max-height: 240px; border-radius: 8px;">

**Conversation Summarization** — when there's nothing left to offload and context hits ~85% of `max_input_tokens`, history is summarized. Full messages move to `/conversation_history/`; a structured summary replaces them in working memory.


In [ ]:
from langchain.agents.middleware import wrap_model_call, wrap_tool_call
from langchain_core.messages import SystemMessage

audit_log = []

@wrap_model_call
def compliance_rules(request, handler):
    """Inject in-store assistant policy into every LLM call."""
    rules = """## In-Store Assistant Policy
- Never store or expose a shopper's full payment card number or account password
- State aisle and stock from store_directory; don't guess a location if it's unknown
- Flag allergens clearly and note when an item is out of stock before suggesting a swap"""
    existing = request.system_message
    blocks = list(existing.content_blocks) if existing else []
    blocks.append({"type": "text", "text": f"\n\n{rules}"})
    return handler(request.override(system_message=SystemMessage(content_blocks=blocks)))

@wrap_tool_call
def audit_trail(request, handler):
    """Create an audit log entry for every tool call."""
    entry = {"tool": request.tool_call["name"], "timestamp": datetime.now().isoformat()}
    result = handler(request)
    entry["status"] = "success"
    audit_log.append(entry)
    return result

agent_with_middleware = create_deep_agent(
    model=model,
    tools=[store_directory, web_search],
    system_prompt="You are an in-store shopping assistant.",
    middleware=[compliance_rules, audit_trail],
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = agent_with_middleware.invoke({
    "messages": [{"role": "user", "content": "Ground beef is out of stock. What aisle is it normally in, and what's a good in-store substitution for tacos? Write the answer to /substitution.md"}]
}, config=config)

print(result["messages"][-1].text)
print(f"\n--- Audit Log ({len(audit_log)} entries) ---")
for entry in audit_log:
    print(f"  {entry['timestamp']}  {entry['tool']}  {entry['status']}")

## 1.6 HITL: Tool-Level Approval

Deep Agents supports `interrupt_on` — pause execution when specific tools are called. The human can approve, edit, or reject.

<img src="../images/deepAgentHITL.png" style="width: auto; max-height: 380px; border-radius: 8px;">


In [ ]:
agent_with_hitl = create_deep_agent(
    model=model,
    tools=[store_directory, web_search],
    system_prompt="You are an in-store shopping assistant.",
    checkpointer=MemorySaver(),
    interrupt_on={
        "write_file": True,
        "edit_file": True,
    },
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = agent_with_hitl.invoke({
    "messages": [{"role": "user", "content": "Write a file called /test.md with 'Hello World'"}]
}, config=config)

if result.get("__interrupt__"):
    interrupt_info = result["__interrupt__"][0].value
    for action in interrupt_info["action_requests"]:
        print(f"Paused — tool: {action['name']}, args: {action['args']}")
    print("\nWaiting for approval...")
    

In [ ]:
# Approve and continue
if result.get("__interrupt__"):
    result = agent_with_hitl.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config,
    )
    print("Approved!")
    print("Agent reply:", result["messages"][-1].text)
    print()
    print_files(result)


## 1.7 AGENTS.md & Skills

`AGENTS.md` replaces hardcoded system prompts with an editable identity file. Skills are loaded on demand — the agent reads them only when the task matches. Here the skill is a **store-route** format that orders the list by aisle so the shopper walks the store once.

In [ ]:
from deepagents.backends.utils import create_file_data

agents_md = """# In-Store Shopping Assistant

You are an in-store shopping assistant helping a shopper move through the store.

## Workflow
1. Plan with write_todos
2. For each item, delegate to item-finder-agent via task() (aisle, stock, price, substitution)
3. Assemble an aisle-ordered route so the shopper walks the store once
4. Save to /store_route.md

## Rules
- Use store_directory for aisle/stock/price; use web search for recipes and swaps
- Order the route by aisle number, not by list order
- Flag out-of-stock items and offer one substitution each
- Check /skills/ for the route format
"""

store_route_skill = """---
name: store-route
description: Turn a shopping list into an aisle-ordered walking route. Use when asked for a store route or to organize a list for shopping.
---

# Store Route Skill

- Start with the store name and item count
- Group items by aisle, in ascending aisle order (Produce first, checkout last)
- For each item: name, price, and a checkbox
- Put OUT-OF-STOCK items in their own section with a suggested substitution
- Keep it scannable on a phone while walking
"""

# The agent's identity lives in the /AGENTS.md file (seeded below) and is loaded
# via the `memory` parameter — no redundant `system_prompt` string.
agent = create_deep_agent(
    model=model,
    tools=[store_directory, web_search],
    subagents=[item_finder_subagent],
    memory=["/AGENTS.md"],
    skills=["/skills/"],
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": uuid7()}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "I'm shopping for tacos: tortillas, ground beef, salsa, shredded cheese, lettuce. Build me an aisle-ordered route and handle anything out of stock. Do not call write_todos, just do it."}],
    "files": {
        "/AGENTS.md": create_file_data(agents_md),
        "/skills/store-route/SKILL.md": create_file_data(store_route_skill),
    },
}, config=config)

for m in result["messages"]:
    m.model_copy(update={"content": truncate(m.text)}).pretty_print()

## 1.8 The Complete Agent

All pieces together: tools, subagents, memory, middleware, HITL, AGENTS.md, and skills.

> `agents/research_agent.py` packages a minimal slice of this (no HITL, no FilesystemBackend) for Module 4's evals.


In [ ]:
store = open_memory_store()  # same SQLite-backed long-term memory as section 1.4
audit_log = []  # reset

complete_backend = CompositeBackend(
    default=StateBackend(),
    routes={
        "/memories/": StoreBackend(
            store=store,
            namespace=lambda rt: ("memories", "shared"),
        ),
    },
)

complete_agent = create_deep_agent(
    model=model,
    tools=[store_directory, web_search],
    subagents=[item_finder_subagent],
    backend=complete_backend,
    store=store,
    middleware=[compliance_rules, audit_trail],
    checkpointer=MemorySaver(),
    interrupt_on={"write_file": True, "edit_file": True},
    memory=["/AGENTS.md"],
    skills=["/skills/"],
)

print("Complete agent created with:")
print("  - Store directory tool + native web search")
print("  - Subagents (item-finder-agent)")
print("  - Memory (/memories/ -> StoreBackend on SQLite)")
print("  - Middleware (in-store policy + audit trail)")
print("  - HITL (interrupt on file writes)")
print("  - AGENTS.md + Skills")

In [ ]:
# Drive the complete agent end-to-end.
# Exercises: subagent delegation, file writes (HITL-gated),
# /memories/ persistence, and middleware (audit + in-store policy).
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": str(uuid7())}}

# Seed AGENTS.md + the store-route skill (same content as the 1.7 cell)
# so the agent has its identity and capabilities loaded.
seed_files = {
    "/AGENTS.md": create_file_data(agents_md),
    "/skills/store-route/SKILL.md": create_file_data(store_route_skill),
}

result = complete_agent.invoke({
    "messages": [HumanMessage(content=(
        "I just walked into my usual store to shop for taco night. "
        "Follow your AGENTS.md workflow: find each item, build the aisle-ordered route to /store_route.md, "
        "and save any new shopper details to /memories/shopper_notes.md. "
        "Use web search at most once (only if something's out of stock)."
    ))],
    "files": seed_files,
}, config=config)

# The HITL middleware pauses on every write_file / edit_file. Approve them all.
while result.get("__interrupt__"):
    payload = result["__interrupt__"][0].value
    actions = payload.get("action_requests", [])
    for action in actions:
        print(f"  HITL pause -> approving {action['name']}: {action['args'].get('file_path','?')}")
    result = complete_agent.invoke(
        Command(resume={"decisions": [{"type": "approve"} for _ in actions]}),
        config=config,
    )

print("\nFinal reply:\n", result["messages"][-1].text[:600])
print()
# Show only files the agent wrote (skip the seed files we passed in).
seed_paths = set(seed_files.keys())
agent_files = {k: v for k, v in (result.get("files") or {}).items() if k not in seed_paths}
print_files({"files": agent_files}, header="FILES THE AGENT WROTE")

print(f"\nAudit log: {len(audit_log)} tool call(s) recorded by the audit middleware")
for entry in audit_log:
    print(f"  {entry['timestamp']}  {entry['tool']:20s} {entry['status']}")

### Deep Agents Recap

| Feature | How | Built-in? |
|---------|-----|----------|
| **Harness** | `create_deep_agent()` | Filesystem, Planning, Summarization |
| **Custom tools** | `tools=[your_tool]` | Added to built-in tools |
| **Subagents** | `subagents=[{name, description, ...}]` | `task()` tool |
| **Memory** | `CompositeBackend` routing to `StoreBackend` | Path-based routing |
| **Middleware** | `middleware=[wrap_model_call, wrap_tool_call]` | Appended to built-in stack |
| **HITL** | `interrupt_on={"write_file": True}` | Configurable per tool |
| **AGENTS.md** | `memory=["/AGENTS.md"]` or `files={}` | Editable identity |
| **Skills** | `skills=["./skills/"]` or `files={}` | On-demand capabilities |